# Texas 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Texas, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `lib_general_total`, `wri_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [2]:
# TX 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/TX/20080304__tx__primary__county.csv"
GENERAL_PATH = r"../../data/raw/2008/TX/20081104__tx__general__county.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/TX/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [3]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,office,district,candidate,incumbent,party,votes,pct
0,ANDERSON,President/VicePresident,NaN,Joe Biden,False,DEM,11,0.21
1,ANDERSON,President/VicePresident,NaN,Hillary Clinton,False,DEM,2975,56.08
2,ANDERSON,President/VicePresident,NaN,Christopher J. Dodd,False,DEM,9,0.17
3,ANDERSON,President/VicePresident,NaN,John Edwards,False,DEM,87,1.64
4,ANDERSON,President/VicePresident,NaN,Barack Obama,False,DEM,2182,41.13
5,ANDERSON,President/VicePresident,NaN,Bill Richardson,False,DEM,41,0.77
6,ANDERSON,President/VicePresident,NaN,Total,NaN,NaN,5305,NaN
7,ANDERSON,U.S. Senate,NaN,Gene Kelly,False,DEM,1205,28.35
8,ANDERSON,U.S. Senate,NaN,Ray McMurrey,False,DEM,772,18.16
9,ANDERSON,U.S. Senate,NaN,Richard J. (Rick) Noriega,False,DEM,1879,44.20


In [4]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President/VicePresident                                         4826
President Congressional                                         3528
Judge Court of Criminal Appeals                                 2793
Justice Supreme Court                                           2539
U.S. Senate                                                     2032
President Senatorial                                            1981
Railroad Commissioner                                           1524
U. S. Representative                                            1200
State Representative                                            1089
Chief Justice, Supreme Court                                    1016
District Judge                                                   864
Prop 2. Require photo ID to cast ballot in elections.            762
Prop 3. Need voter approval to exceed allowed annual growth.     762
Prop 1. Officials to enforce U.S. immigration laws.              762
District Attorney          

In [5]:
# Only keep rows where 'office' is 'President/VicePresident'
primary_df = primary_df[primary_df["office"] == "President/VicePresident"]
primary_df.head(DISPLAY_ROWS)

,county,office,district,candidate,incumbent,party,votes,pct
0,ANDERSON,President/VicePresident,NaN,Joe Biden,False,DEM,11,0.21
1,ANDERSON,President/VicePresident,NaN,Hillary Clinton,False,DEM,2975,56.08
2,ANDERSON,President/VicePresident,NaN,Christopher J. Dodd,False,DEM,9,0.17
3,ANDERSON,President/VicePresident,NaN,John Edwards,False,DEM,87,1.64
4,ANDERSON,President/VicePresident,NaN,Barack Obama,False,DEM,2182,41.13
5,ANDERSON,President/VicePresident,NaN,Bill Richardson,False,DEM,41,0.77
6,ANDERSON,President/VicePresident,NaN,Total,NaN,NaN,5305,NaN
35,ANDREWS,President/VicePresident,NaN,Joe Biden,False,DEM,10,0.80
36,ANDREWS,President/VicePresident,NaN,Hillary Clinton,False,DEM,674,53.79
37,ANDREWS,President/VicePresident,NaN,Christopher J. Dodd,False,DEM,10,0.80


In [6]:
# Primary data shape when only considering President/VicePresident
primary_df.shape

(4826, 8)

In [7]:
# Number of missing values in each column
primary_df.isna().sum()

county          0
office          0
district     4826
candidate       0
incumbent     509
party         509
votes           0
pct           509
dtype: int64

In [8]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,candidate,incumbent,party,votes,pct
0,ANDERSON,Joe Biden,False,DEM,11,0.21
1,ANDERSON,Hillary Clinton,False,DEM,2975,56.08
2,ANDERSON,Christopher J. Dodd,False,DEM,9,0.17
3,ANDERSON,John Edwards,False,DEM,87,1.64
4,ANDERSON,Barack Obama,False,DEM,2182,41.13
5,ANDERSON,Bill Richardson,False,DEM,41,0.77
6,ANDERSON,Total,NaN,NaN,5305,NaN
7,ANDREWS,Joe Biden,False,DEM,10,0.80
8,ANDREWS,Hillary Clinton,False,DEM,674,53.79
9,ANDREWS,Christopher J. Dodd,False,DEM,10,0.80


**Preprocessing notes for `primary_df`:**

* We currently keep only rows where `office == "President/VicePresident"`. In the 2008 Texas primaries, the ballot line labeled “President/Vice-President” is the presidential primary. Vice-presidential nominees are not chosen in state primaries. They’re selected later by party conventions.

* Some rows have `candidate == "Total"`, which are county-level aggregates for Texas (254 counties). These sum rows should be dropped.

* The `incumbent` column is not needed for our analysis, so we will drop it.

* Once we restrict the data to presidential candidates, percentage shares will change. We will drop the `pct` column and recompute percentages after filtering.

* There are missing values in a large number of rows in `party` that we need to figure a way to fix this.

In [9]:
# Drop the `incumbent` and `pct` columns
primary_df = primary_df.drop(columns=["incumbent", "pct"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Joe Biden,DEM,11
1,ANDERSON,Hillary Clinton,DEM,2975
2,ANDERSON,Christopher J. Dodd,DEM,9
3,ANDERSON,John Edwards,DEM,87
4,ANDERSON,Barack Obama,DEM,2182
5,ANDERSON,Bill Richardson,DEM,41
6,ANDERSON,Total,NaN,5305
7,ANDREWS,Joe Biden,DEM,10
8,ANDREWS,Hillary Clinton,DEM,674
9,ANDREWS,Christopher J. Dodd,DEM,10


In [10]:
# Drop rows where "candidate" == "Total"
primary_df = primary_df[primary_df['candidate'] != 'Total'].reset_index(drop=True)
primary_df.shape

(4318, 4)

In [11]:
# Missing values now in each column
primary_df.isna().sum()

county       0
candidate    0
party        1
votes        0
dtype: int64

There is a single observation left that has missing value in `party`. We might want to explicitly check this row out.

In [12]:
# Observation with missing value in `party`
primary_df.loc[primary_df["party"].isna()]

,county,candidate,party,votes
2678,HAYS,Uncommitted,NaN,REP


This is just an uncommitted candidate with no party and no value in the number of votes. We can drop this observation and have a dataframe that don't have any more missing values.

In [13]:
# Drop observation with misisng value in `party`
primary_df = primary_df[
    ~((primary_df["candidate"] == "Uncommitted") & (primary_df["party"].isna()))
].reset_index(drop=True)

# Shape of primary_df after dropping such observation
primary_df.shape

(4317, 4)

In [14]:
# Quick peek of the current primary_df
primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Joe Biden,DEM,11
1,ANDERSON,Hillary Clinton,DEM,2975
2,ANDERSON,Christopher J. Dodd,DEM,9
3,ANDERSON,John Edwards,DEM,87
4,ANDERSON,Barack Obama,DEM,2182
5,ANDERSON,Bill Richardson,DEM,41
6,ANDREWS,Joe Biden,DEM,10
7,ANDREWS,Hillary Clinton,DEM,674
8,ANDREWS,Christopher J. Dodd,DEM,10
9,ANDREWS,John Edwards,DEM,82


In [15]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Joe Biden              254
Duncan Hunter          254
Hoa Tran               254
Fred Thompson          254
Mitt Romney            254
Ron Paul               254
John McCain            254
Alan Keyes             254
Mike Huckabee          254
Hillary Clinton        254
Rudy Giuliani          254
Hugh Cort              254
Bill Richardson        254
Barack Obama           254
John Edwards           254
Christopher J. Dodd    254
Uncommitted            253
Name: count, dtype: int64

In [16]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
REP    2793
DEM    1524
Name: count, dtype: int64

In [17]:
# Data type of each column in primary_df
primary_df.dtypes

county       object
candidate    object
party        object
votes        object
dtype: object

Note that, for `votes`, we expect the data type to be integer ("Int64"). Thus, we will coerce it not to avoid any future error.

In [18]:
# Coerce "votes" to have integer type
primary_df["votes"] = pd.to_numeric(primary_df["votes"], errors="coerce")

# Check the data type again
primary_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [19]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Joe Biden,DEM,11
1,ANDERSON,Hillary Clinton,DEM,2975
2,ANDERSON,Christopher J. Dodd,DEM,9
3,ANDERSON,John Edwards,DEM,87
4,ANDERSON,Barack Obama,DEM,2182
5,ANDERSON,Bill Richardson,DEM,41
6,ANDREWS,Joe Biden,DEM,10
7,ANDREWS,Hillary Clinton,DEM,674
8,ANDREWS,Christopher J. Dodd,DEM,10
9,ANDREWS,John Edwards,DEM,82


In [20]:
# Shape after preprocessing
primary_df.shape

(4317, 4)

### b. General Election Dataset

In [21]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,candidate,incumbent,party,votes,pct
0,ANDERSON,President/VicePresident,NaN,John McCain/ Sarah Palin,False,REP,11884,71.35
1,ANDERSON,President/VicePresident,NaN,Barack Obama/ Joe Biden,False,DEM,4630,27.80
2,ANDERSON,President/VicePresident,NaN,Bob Barr/ Wayne A. Root,False,LIB,115,0.69
3,ANDERSON,President/VicePresident,NaN,Chuck Baldwin/ Darrell L. Castle,False,WI,10,0.06
4,ANDERSON,President/VicePresident,NaN,Thaddaus Hill/ Gordon F. Bailey,False,WI,0,0.00
5,ANDERSON,President/VicePresident,NaN,Jonathan Allen/ Jeffrey D. Stath,False,WI,0,0.00
6,ANDERSON,President/VicePresident,NaN,"Alan Keyes/ Marvin Sprouse, Jr.",False,WI,5,0.03
7,ANDERSON,President/VicePresident,NaN,Ralph Nader/ Matt Gonzalez,False,WI,11,0.07
8,ANDERSON,President/VicePresident,NaN,Cynthia McKinney/ Rosa Clemente,False,WI,0,0.00
9,ANDERSON,President/VicePresident,NaN,Brian Moore/ Stewart A. Alexander,False,WI,0,0.00


In [22]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President/VicePresident                              2794
Judge Court of Criminal Appeals                      2794
Justice Supreme Court                                2032
U. S. Representative                                 1043
State Representative                                 1024
Railroad Commissioner                                1016
Chief Justice, Supreme Court                         1016
U.S. Senate                                          1016
District Judge                                        689
State Senate                                          459
District Attorney                                     379
Member State Board of Education                       286
Chief Justice, 4th Court of Appeals District           96
Chief Justice, 7th Court of Appeals District           92
Justice 14th Court of Appeals District                 90
Chief Justice, 3rd Court of Appeals District           72
Justice 13th Court of Appeals District                 60
Justice

In [23]:
# Only keep rows where 'office' is 'President/VicePresident'
general_df = general_df[general_df["office"] == "President/VicePresident"]
general_df.head(DISPLAY_ROWS)

,county,office,district,candidate,incumbent,party,votes,pct
0,ANDERSON,President/VicePresident,NaN,John McCain/ Sarah Palin,False,REP,11884,71.35
1,ANDERSON,President/VicePresident,NaN,Barack Obama/ Joe Biden,False,DEM,4630,27.80
2,ANDERSON,President/VicePresident,NaN,Bob Barr/ Wayne A. Root,False,LIB,115,0.69
3,ANDERSON,President/VicePresident,NaN,Chuck Baldwin/ Darrell L. Castle,False,WI,10,0.06
4,ANDERSON,President/VicePresident,NaN,Thaddaus Hill/ Gordon F. Bailey,False,WI,0,0.00
5,ANDERSON,President/VicePresident,NaN,Jonathan Allen/ Jeffrey D. Stath,False,WI,0,0.00
6,ANDERSON,President/VicePresident,NaN,"Alan Keyes/ Marvin Sprouse, Jr.",False,WI,5,0.03
7,ANDERSON,President/VicePresident,NaN,Ralph Nader/ Matt Gonzalez,False,WI,11,0.07
8,ANDERSON,President/VicePresident,NaN,Cynthia McKinney/ Rosa Clemente,False,WI,0,0.00
9,ANDERSON,President/VicePresident,NaN,Brian Moore/ Stewart A. Alexander,False,WI,0,0.00


In [24]:
# Primary data shape when only considering President/VicePresident
general_df.shape

(2794, 8)

In [25]:
# Number of missing values in each column
general_df.isna().sum()

county          0
office          0
district     2794
candidate       0
incumbent     254
party         254
votes           0
pct           254
dtype: int64

In [26]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
# Also the "incumbent" and "pct" as we don't need it in our future analysis
general_df = general_df.drop(columns=["office", "district", "incumbent", "pct"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,John McCain/ Sarah Palin,REP,11884
1,ANDERSON,Barack Obama/ Joe Biden,DEM,4630
2,ANDERSON,Bob Barr/ Wayne A. Root,LIB,115
3,ANDERSON,Chuck Baldwin/ Darrell L. Castle,WI,10
4,ANDERSON,Thaddaus Hill/ Gordon F. Bailey,WI,0
5,ANDERSON,Jonathan Allen/ Jeffrey D. Stath,WI,0
6,ANDERSON,"Alan Keyes/ Marvin Sprouse, Jr.",WI,5
7,ANDERSON,Ralph Nader/ Matt Gonzalez,WI,11
8,ANDERSON,Cynthia McKinney/ Rosa Clemente,WI,0
9,ANDERSON,Brian Moore/ Stewart A. Alexander,WI,0


In [27]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
John McCain/ Sarah Palin             254
Barack Obama/ Joe Biden              254
Bob Barr/ Wayne A. Root              254
Chuck Baldwin/ Darrell L. Castle     254
Thaddaus Hill/ Gordon F. Bailey      254
Jonathan Allen/ Jeffrey D. Stath     254
Alan Keyes/ Marvin Sprouse, Jr.      254
Ralph Nader/ Matt Gonzalez           254
Cynthia McKinney/ Rosa Clemente      254
Brian Moore/ Stewart A. Alexander    254
Total                                254
Name: count, dtype: int64

We also drop rows where values in `candidate` column are "Total".


In [28]:
# Drop rows where "candidate" == "Total"
general_df = general_df[general_df['candidate'] != 'Total'].reset_index(drop=True)
general_df.shape

(2540, 4)

Now, each row’s candidate value now contains two names: presidential first, vice-presidential second, which is separated by a backslash ("\"). We’ll split on the backslash and retain only the presidential name.

In [29]:
# Keep only the presidential candidate in the "candidate" column
general_df["candidate"] = (
    general_df["candidate"]
      .str.split(r"(?i)\s*(?:andf|and|&|/|/)\s*", n=1, expand=True)[0]
      .str.strip()
)

# Candidates in general_df
general_df["candidate"].value_counts()

candidate
John McCain         254
Barack Obama        254
Bob Barr            254
Chuck Baldwin       254
Thaddaus Hill       254
Jonathan Allen      254
Alan Keyes          254
Ralph Nader         254
Cynthia McKinney    254
Brian Moore         254
Name: count, dtype: int64

In [30]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
WI     1778
REP     254
DEM     254
LIB     254
Name: count, dtype: int64

In [31]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [32]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,John McCain,REP,11884
1,ANDERSON,Barack Obama,DEM,4630
2,ANDERSON,Bob Barr,LIB,115
3,ANDERSON,Chuck Baldwin,WI,10
4,ANDERSON,Thaddaus Hill,WI,0
5,ANDERSON,Jonathan Allen,WI,0
6,ANDERSON,Alan Keyes,WI,5
7,ANDERSON,Ralph Nader,WI,11
8,ANDERSON,Cynthia McKinney,WI,0
9,ANDERSON,Brian Moore,WI,0


In [33]:
# Shape after preprocessing
general_df.shape

(2540, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [34]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Wi" : "wri",                       # Three-letter abbreviation format
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [35]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [36]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [37]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_KEYES,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_THOMPSON,pri_rep_TRAN,pri_rep_UNCOMMITTED
0,ANDERSON,11,2975,9,87,2182,41,1,8,1816,9,12,2101,156,34,28,0,38
1,ANDREWS,10,674,10,82,448,29,1,4,446,2,5,480,15,13,14,1,2
2,ANGELINA,53,7714,33,387,4329,70,12,20,2389,12,14,2254,146,69,39,2,28
3,ARANSAS,2,1402,1,22,1018,14,1,33,882,9,15,2667,300,86,40,7,155
4,ARCHER,7,785,6,36,296,12,1,9,458,5,3,742,26,20,8,1,16
5,ARMSTRONG,0,0,0,0,0,0,0,4,226,4,4,194,59,5,5,1,23
6,ATASCOSA,16,3451,15,101,1971,35,2,8,740,2,8,923,54,21,16,0,11
7,AUSTIN,2,1745,1,20,1218,5,0,6,1166,69,23,1413,159,46,17,0,33
8,BAILEY,2,306,0,14,175,10,0,2,266,0,0,376,13,9,4,0,12
9,BANDERA,1,954,4,15,783,10,3,25,1155,12,41,2312,241,73,41,2,169


Note that there is a column with uncommitted candidate that still had votes (`pri_rep_UNCOMMITTED`). We will keep this for total counting purposes and drop them at the end.

In [38]:
# Primary dataframe shape after pivot
primary_pivot.shape

(254, 18)

In [39]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_lib_BARR,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_HILL,gen_wri_KEYES,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER
0,ANDERSON,4630,115,11884,0,10,0,5,0,0,11
1,ANDREWS,790,18,3816,1,4,0,0,0,0,0
2,ANGELINA,9379,189,19569,0,9,0,1,0,0,6
3,ARANSAS,3006,79,6693,0,0,0,0,0,0,0
4,ARCHER,740,26,3595,0,1,0,0,0,0,3
5,ARMSTRONG,128,6,856,0,0,0,0,0,0,0
6,ATASCOSA,4415,47,5462,0,3,0,2,0,0,9
7,AUSTIN,2821,72,8786,0,6,19,4,1,0,12
8,BAILEY,682,14,1618,1,0,0,0,0,0,1
9,BANDERA,2250,83,6935,0,21,0,0,2,0,6


In [40]:
# General dataframe shape after pivot
general_pivot.shape

(254, 11)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [41]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 254 out of 254


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [42]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_GIULIANI,pri_rep_HUCKABEE,...,gen_dem_OBAMA,gen_lib_BARR,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_HILL,gen_wri_KEYES,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER
0,ANDERSON,11,2975,9,87,2182,41,1,8,1816,...,4630,115,11884,0,10,0,5,0,0,11
1,ANDREWS,10,674,10,82,448,29,1,4,446,...,790,18,3816,1,4,0,0,0,0,0
2,ANGELINA,53,7714,33,387,4329,70,12,20,2389,...,9379,189,19569,0,9,0,1,0,0,6
3,ARANSAS,2,1402,1,22,1018,14,1,33,882,...,3006,79,6693,0,0,0,0,0,0,0
4,ARCHER,7,785,6,36,296,12,1,9,458,...,740,26,3595,0,1,0,0,0,0,3
5,ARMSTRONG,0,0,0,0,0,0,0,4,226,...,128,6,856,0,0,0,0,0,0,0
6,ATASCOSA,16,3451,15,101,1971,35,2,8,740,...,4415,47,5462,0,3,0,2,0,0,9
7,AUSTIN,2,1745,1,20,1218,5,0,6,1166,...,2821,72,8786,0,6,19,4,1,0,12
8,BAILEY,2,306,0,14,175,10,0,2,266,...,682,14,1618,1,0,0,0,0,0,1
9,BANDERA,1,954,4,15,783,10,3,25,1155,...,2250,83,6935,0,21,0,0,2,0,6


In [43]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,...,gen_dem_OBAMA,gen_lib_BARR,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_HILL,gen_wri_KEYES,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER
count,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,...,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000,254.000000
mean,20.826772,5758.795276,14.870079,117.858268,5364.078740,42.413386,2.866142,23.771654,2039.377953,32.370079,...,13892.255906,220.929134,17635.149606,0.409449,22.472441,0.850394,3.523622,3.578740,0.531496,22.641732
std,39.159771,17749.409335,30.577880,161.450599,21884.974571,91.734744,12.170958,63.870446,5731.647269,208.964123,...,55336.186566,715.861135,52423.284118,1.457456,75.721387,4.111257,11.550774,20.242007,1.987359,86.998487
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,8.000000,0.000000,67.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,502.250000,2.000000,27.250000,242.250000,9.000000,0.000000,1.000000,137.000000,1.000000,...,626.250000,16.250000,1488.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,8.000000,1472.500000,7.000000,65.000000,855.000000,20.000000,0.000000,6.000000,458.500000,4.000000,...,1914.000000,42.500000,4208.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,3.000000
75%,21.000000,3477.250000,15.000000,141.250000,2179.000000,41.000000,2.000000,20.750000,1495.250000,14.000000,...,5252.750000,109.500000,11880.750000,0.000000,14.000000,0.000000,2.000000,1.000000,0.000000,11.750000
max,270.000000,176268.000000,260.000000,1271.000000,228610.000000,885.000000,160.000000,715.000000,57945.000000,3194.000000,...,590982.000000,6783.000000,571883.000000,12.000000,667.000000,42.000000,102.000000,253.000000,16.000000,858.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `lib_general_total` = sum of all `gen_lib_*` columns
    * `wri_general_total` = sum of all `gen_wri_*` columns

In [ ]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the UNCOMMITTED column.

In [ ]:
# Drop UNCOMMITTED column for primary election
merged_df = merged_df.drop(columns="pri_rep_UNCOMMITTED")

# Snippet at the merged dataframe with primary totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_GIULIANI,pri_rep_HUCKABEE,...,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_HILL,gen_wri_KEYES,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,rep_primary_total,dem_primary_total
0,ANDERSON,11,2975,9,87,2182,41,1,8,1816,...,11884,0,10,0,5,0,0,11,4203,5305
1,ANDREWS,10,674,10,82,448,29,1,4,446,...,3816,1,4,0,0,0,0,0,983,1253
2,ANGELINA,53,7714,33,387,4329,70,12,20,2389,...,19569,0,9,0,1,0,0,6,4985,12586
3,ARANSAS,2,1402,1,22,1018,14,1,33,882,...,6693,0,0,0,0,0,0,0,4195,2459
4,ARCHER,7,785,6,36,296,12,1,9,458,...,3595,0,1,0,0,0,0,3,1289,1142
5,ARMSTRONG,0,0,0,0,0,0,0,4,226,...,856,0,0,0,0,0,0,0,525,0
6,ATASCOSA,16,3451,15,101,1971,35,2,8,740,...,5462,0,3,0,2,0,0,9,1785,5589
7,AUSTIN,2,1745,1,20,1218,5,0,6,1166,...,8786,0,6,19,4,1,0,12,2932,2991
8,BAILEY,2,306,0,14,175,10,0,2,266,...,1618,1,0,0,0,0,0,1,682,507
9,BANDERA,1,954,4,15,783,10,3,25,1155,...,6935,0,21,0,0,2,0,6,4074,1767


In [47]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
lib_general_cols   = [c for c in merged_df.columns if c.startswith("gen_lib_")]
wri_general_cols   = [c for c in merged_df.columns if c.startswith("gen_wri_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["lib_general_total"] = merged_df[lib_general_cols].sum(axis=1) if lib_general_cols else 0
merged_df["wri_general_total"] = merged_df[wri_general_cols].sum(axis=1) if wri_general_cols else 0

In [48]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_OBAMA', 'pri_dem_RICHARDSON',
       'pri_rep_CORT', 'pri_rep_GIULIANI', 'pri_rep_HUCKABEE',
       'pri_rep_HUNTER', 'pri_rep_KEYES', 'pri_rep_MCCAIN', 'pri_rep_PAUL',
       'pri_rep_ROMNEY', 'pri_rep_THOMPSON', 'pri_rep_TRAN', 'gen_dem_OBAMA',
       'gen_lib_BARR', 'gen_rep_MCCAIN', 'gen_wri_ALLEN', 'gen_wri_BALDWIN',
       'gen_wri_HILL', 'gen_wri_KEYES', 'gen_wri_MCKINNEY', 'gen_wri_MOORE',
       'gen_wri_NADER', 'rep_primary_total', 'dem_primary_total',
       'rep_general_total', 'dem_general_total', 'lib_general_total',
       'wri_general_total'],
      dtype='object')

In [49]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_CORT,pri_rep_GIULIANI,pri_rep_HUCKABEE,...,gen_wri_KEYES,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,lib_general_total,wri_general_total
0,ANDERSON,11,2975,9,87,2182,41,1,8,1816,...,5,0,0,11,4203,5305,11884,4630,115,26
1,ANDREWS,10,674,10,82,448,29,1,4,446,...,0,0,0,0,983,1253,3816,790,18,5
2,ANGELINA,53,7714,33,387,4329,70,12,20,2389,...,1,0,0,6,4985,12586,19569,9379,189,16
3,ARANSAS,2,1402,1,22,1018,14,1,33,882,...,0,0,0,0,4195,2459,6693,3006,79,0
4,ARCHER,7,785,6,36,296,12,1,9,458,...,0,0,0,3,1289,1142,3595,740,26,4
5,ARMSTRONG,0,0,0,0,0,0,0,4,226,...,0,0,0,0,525,0,856,128,6,0
6,ATASCOSA,16,3451,15,101,1971,35,2,8,740,...,2,0,0,9,1785,5589,5462,4415,47,14
7,AUSTIN,2,1745,1,20,1218,5,0,6,1166,...,4,1,0,12,2932,2991,8786,2821,72,42
8,BAILEY,2,306,0,14,175,10,0,2,266,...,0,0,0,1,682,507,1618,682,14,2
9,BANDERA,1,954,4,15,783,10,3,25,1155,...,0,2,0,6,4074,1767,6935,2250,83,29


Now, we save the cleaned dataframe into the processed directory.

In [50]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "TX.csv", index=False)